# M8 lab: deciding before the uncertainty resolves

*Energy Systems, Step by Step* · Modeler tier · lab for lesson M8 · about 60 minutes, plus 90 minutes of practice at the end.

You will rebuild the lesson's toy as linear programs, solve them with the open-source HiGHS solver through `scipy.optimize.linprog(method="highs")`, and check your results against the lesson's reference block with `assert` cells:

1. The wind blocks: a year of hours from the 2025 wind shape, cut into five blocks.
2. The cost of a size in each branch, in closed form.
3. One builder for every linear program of the lesson: the extensive form, wait-and-see, expected-value, a plan priced with its size fixed, worst case and least regret.
4. The plans, the value of the stochastic solution (VSS) and the expected value of perfect information (EVPI).
5. Robust plans, and a check that every size is the only best one.
6. A bigger tree: nine branches, two uncertainties.

**How to run it** (it runs in seconds):

- Put this notebook and the three data files in one folder.
- **In Google Colab:** *File → Upload notebook*, upload the files with the folder icon on the left, then *Runtime → Run all*.
- **Locally:** JupyterLab, or VS Code with its Jupyter extension, and Python 3.9 or later with `numpy` and `scipy` 1.9 or later. Start it from that folder: the files are read from the working folder.

| File (download from the lesson page) | What it holds |
|---|---|
| `m08-stochastic-robust.json` | the windy zone, the three build-outs and their chances, the wind blocks, the electrolyzer and the hydrogen, and the reference results |
| `town_year.json` | the 8,760 hourly shapes of 2025 (Southwest Power Pool, EIA-930), in thousandths of each series' maximum |
| `cf_targets.json` | EIA's 2025 capacity factors (wind 33.8%) |

**The data.** The zone, the chances, the costs and the demand are fictional, illustrative round numbers in 2025 dollars; the wind blocks come from a real 2025 hourly shape. Nothing here is a study's input or result.

**What the costs leave out.** Surplus wind would otherwise be curtailed, so the electrolyzer takes it at no charge. Gas reformers supply any amount at \$1.50 per kg. The wind and the lines cost the same whatever the size, so they are left out.

In [ ]:
import json
from pathlib import Path

import numpy as np
import scipy
from scipy.optimize import linprog
from scipy.sparse import coo_matrix

major, minor = (int(x) for x in scipy.__version__.split(".")[:2])
assert (major, minor) >= (1, 9), "this lab needs SciPy 1.9 or later (pip install -U scipy)"
FILES = ["m08-stochastic-robust.json", "town_year.json", "cf_targets.json"]
for name in FILES:
    assert Path(name).exists(), f"{name} not found: put it next to this notebook (in Colab, upload it with the folder icon)."
D, year, targets = (json.load(open(n)) for n in FILES)
REF = D["reference"]
print("NumPy", np.__version__, "· SciPy", scipy.__version__)
print(year["meta"]["source"])

## 1. The wind blocks

The wind shape $u_t$ (thousandths of its 2025 maximum) is scaled so the year averages EIA's 33.8%: output per MW $= \min(1, k\,u_t)$, with $k$ found by bisection (the lesson's method "scale"; the windiest hour stays below capacity, so nothing is clipped). Sorted from the windiest hour down (a duration curve) and cut at 500, 1,500, 3,000 and 5,000 hours, each block's mean output is its capacity factor $f_b$.

In [ ]:
u = np.maximum(np.asarray(year["wind"], float), 0) / 1000
target = targets["targets"]["wind"]["cf"]
cf_of = lambda k: np.minimum(1, k * u).mean()
lo, hi = 0.0, 1.0
while cf_of(hi) < target:
    hi *= 2
for _ in range(200):
    mid = (lo + hi) / 2
    lo, hi = (mid, hi) if cf_of(mid) < target else (lo, mid)
wind = np.minimum(1, (lo + hi) / 2 * u)
dur = np.sort(wind)[::-1]
edges = [0] + D["blocks"]["edges"]
f = np.array([round(dur[a:b].mean(), 3) for a, b in zip(edges[:-1], edges[1:])])
h = np.diff(edges)
print("capacity factor", round(wind.mean(), 4), "· windiest hour", round(dur[0], 3))
for lab, hb, fb in zip(D["blocks"]["labels"], h, f):
    print(f"{lab:22s} {hb:5d} h at {fb:.3f}")
assert np.allclose(f, D["blocks"]["cf"]) and list(h) == D["blocks"]["hours"]

## 2. The cost of a size, in closed form

- **Surplus.** In branch $s$, with $W_s$ MW of wind and lines that take $A$ = 1,000 MW, the surplus in block $b$ is $\sigma_{s,b} = \max(0, W_s f_b - A)$.
- **What the electrolyzer uses.** An electrolyzer of $x$ MW uses $\min(x, \sigma_{s,b}, H/k)$ in each hour of block $b$. Hydrogen demand caps it too: $H/k$ = 54,000 ÷ 18 = 3,000 MW makes all the hydrogen industry takes.
- **What it saves.** Every MWh it uses saves 18 kg of reformer hydrogen at \$1.50. A year then costs

$$\text{cost}_s(x) = c\,x + c_r\Big(H \cdot 8{,}760 - k \sum_b h_b \min\big(x, \sigma_{s,b}, H/k\big)\Big).$$

- **Its shape.** It is piecewise linear in $x$, with kinks at the surpluses, and convex (a sum of $c\,x$ and $-\min(\cdot)$ terms).
- **The critical hours.** One more MW pays while it runs more than $c / (c_r k)$ hours a year.
- **The cap matters in section 6.** In the three branches the largest surplus, 2,997 MW, stays below 3,000 MW. With a 900 MW line limit, all the planned wind gives 3,097 MW in the windiest block: there the cap binds. A branch may carry its own line limit `A`; `cost` and `expected` take it from the branch.

In [ ]:
A, c = D["zone"]["lineMW"], D["electrolyzer"]["costPerMWYear"]
k, H, cr = D["electrolyzer"]["kgPerMWh"], D["hydrogen"]["demandKgPerHour"], D["hydrogen"]["reformerPerKg"]
f = np.array(D["blocks"]["cf"]); h = np.array(D["blocks"]["hours"])
def branches(p=None):
    return [dict(id=s["id"], short=s["short"], p=(p[i] if p else s["p"]), W=D["wind"]["existingMW"] + s["completion"] * D["wind"]["plannedMW"])
            for i, s in enumerate(D["scenarios"])]
S = branches()
surplus = lambda W, a=A: np.maximum(0, np.round(W * f - a, 6))
# the electrolyzer never runs above H / k = 3,000 MW: demand takes no more
cost = lambda W, x, a=A: c * x + cr * (H * 8760 - k * (h * np.minimum(min(x, H / k), surplus(W, a))).sum())
expected = lambda S, x: sum(s["p"] * cost(s["W"], x, s.get("A", A)) for s in S)
print("critical hours:", round(c / (cr * k)), "· reformers alone: $", f"{cr * H * 8760 / 1e6:.2f}M a year")
for s in S:
    print(f"{s['short']:10s} {s['W']:6,.0f} MW of wind, chance {s['p']:.2f}, surplus by block:", surplus(s["W"]))
assert max(surplus(s["W"]).max() for s in S) < H / k, "in the three branches the demand cap H/k never binds"

## 3. One builder for every linear program

Columns: $x$, then for each branch and block (a branch may carry its own line limit `A`; section 6 uses it) the electrolyzer power $e_{s,b}$ (MW) and reformer hydrogen $r_{s,b}$ (kg an hour); the robust modes add a free $z$. Rows: $k\,e_{s,b} + r_{s,b} = H$ (equalities) and $e_{s,b} - x \le 0$; the robust modes add $c\,x + \sum_b h_b c_r r_{s,b} - z \le 0$ (worst case) or $\le \text{WS}_s$ (least regret). The bound $e_{s,b} \le \sigma_{s,b}$ is a column bound.

In [ ]:
def build(S, mode="expected", fix=None, ws=None):
    B, n = len(f), 1 + (mode != "expected") + 2 * len(f) * len(S)
    cvec = np.zeros(n); lo_b = np.zeros(n); hi_b = np.full(n, np.inf)
    cvec[0] = c if mode == "expected" else 0
    if fix is not None:
        lo_b[0] = hi_b[0] = fix
    zi = 1 if mode != "expected" else None
    if zi:
        cvec[zi] = 1; lo_b[zi] = -np.inf
    col = lambda si, b, j: 1 + (mode != "expected") + 2 * (si * B + b) + j   # j: 0 = e, 1 = r
    ub_r, ub_c, ub_v, b_ub, eq_r, eq_c, eq_v, b_eq = [], [], [], [], [], [], [], []
    for si, s in enumerate(S):
        sg = surplus(s["W"], s.get("A", A))
        for b in range(B):
            e, r = col(si, b, 0), col(si, b, 1)
            hi_b[e] = sg[b]
            if mode == "expected":
                cvec[r] = s["p"] * h[b] * cr
            row = len(b_ub); ub_r += [row, row]; ub_c += [e, 0]; ub_v += [1, -1]; b_ub.append(0)
            row = len(b_eq); eq_r += [row, row]; eq_c += [e, r]; eq_v += [k, 1]; b_eq.append(H)
        if mode != "expected":
            row = len(b_ub)
            ub_r += [row, row]; ub_c += [0, zi]; ub_v += [c, -1]
            for b in range(B):
                ub_r.append(row); ub_c.append(col(si, b, 1)); ub_v.append(h[b] * cr)
            b_ub.append(ws[s["id"]] if mode == "regret" else 0)
    A_ub = coo_matrix((ub_v, (ub_r, ub_c)), shape=(len(b_ub), n)).tocsr()
    A_eq = coo_matrix((eq_v, (eq_r, eq_c)), shape=(len(b_eq), n)).tocsr()
    return dict(c=cvec, A_ub=A_ub, b_ub=np.array(b_ub, float), A_eq=A_eq, b_eq=np.array(b_eq, float), bounds=list(zip(lo_b, hi_b)), col=col)

def solve(S, **kw):
    m = build(S, **kw)
    res = linprog(m["c"], A_ub=m["A_ub"], b_ub=m["b_ub"], A_eq=m["A_eq"], b_eq=m["b_eq"], bounds=m["bounds"], method="highs")
    assert res.status == 0, res.message
    x = 0.0 if abs(res.x[0]) < 1e-7 else round(res.x[0], 6)
    per = {s["id"]: c * x + sum(h[b] * cr * res.x[m["col"](si, b, 1)] for b in range(len(f))) for si, s in enumerate(S)}
    return x, res.fun, per

m0 = build(S)
print("extensive form:", m0["c"].size, "columns,", m0["A_ub"].shape[0] + m0["A_eq"].shape[0], "rows")
assert m0["c"].size == REF["size"]["cols"] and m0["A_ub"].shape[0] + m0["A_eq"].shape[0] == REF["size"]["rows"]

## 4. Wait and see, the plan for the average, and the stochastic program

- **Wait and see:** each branch solved alone, its chance set to 1. Their chance-weighted cost is WS.
- **The plan for the average:** one branch with the mean wind; then its size, fixed, priced in all three branches (EEV).
- **The stochastic program:** the extensive form, one size for all branches (RP).

Then VSS = EEV − RP and EVPI = RP − WS. Costs are asserted to within a dollar.

In [ ]:
WSc = {}
for s in S:
    x, cst, _ = solve([dict(s, p=1)])
    WSc[s["id"]] = cst
    print(f"wait and see, {s['short']:10s}: {x:7.1f} MW  ${cst / 1e6:8.2f}M")
    assert abs(x - REF["ws"][s["id"]]["x"]) < 1e-6 and abs(cst - REF["ws"][s["id"]]["cost"]) <= 1
WS = sum(s["p"] * WSc[s["id"]] for s in S)
Wbar = sum(s["p"] * s["W"] for s in S)
x_ev, own, _ = solve([dict(id="ev", short="average", p=1, W=Wbar)])
_, EEV, per_ev = solve(S, fix=x_ev)
x_sp, RP, per_sp = solve(S)
print(f"plan for the average ({Wbar:,.0f} MW): {x_ev:.1f} MW, expects ${own / 1e6:.2f}M, costs ${EEV / 1e6:.2f}M over the branches")
print(f"stochastic program: {x_sp:.1f} MW, ${RP / 1e6:.2f}M")
print(f"WS ${WS / 1e6:.2f}M <= RP ${RP / 1e6:.2f}M <= EEV ${EEV / 1e6:.2f}M;  VSS ${(EEV - RP) / 1e6:.2f}M, EVPI ${(RP - WS) / 1e6:.2f}M")
assert abs(x_ev - REF["ev"]["x"]) < 1e-6 and abs(EEV - REF["ev"]["exp"]) <= 1 and abs(own - REF["ev"]["own"]) <= 1
assert abs(x_sp - REF["sp"]["x"]) < 1e-6 and abs(RP - REF["sp"]["exp"]) <= 1
assert WS <= RP <= EEV and abs((EEV - RP) - REF["vss"]) <= 2 and abs((RP - WS) - REF["evpi"]) <= 2
# the closed form agrees, and the expected hours an extra MW runs on either side of the plan straddle the critical hours
assert abs(expected(S, x_sp) - RP) <= 1
hours_above = lambda x: sum(s["p"] * h[surplus(s["W"]) > x].sum() for s in S)
print("expected hours just below and above the plan:", hours_above(x_sp - 1e-6), hours_above(x_sp + 1e-6), "· critical", round(c / (cr * k)))

## 5. Robust plans, and is each size the only best one?

The worst-case plan minimizes the largest branch cost; the least-regret plan minimizes the largest regret, a branch's cost minus its wait-and-see cost. Neither uses the chances. Each robust plan is then priced with its size fixed, so its branch costs are unique.

**Is each size the only best one?**

- **The shape.** Every cost here is convex and piecewise linear in the size; a max of convex costs is convex too.
- **The cost of another MW.** It costs $c$ and saves $c_r k$ in every hour of a block whose surplus lies above the size: an exact slope.
- **The slope below** the size must be negative, and **the slope above** positive. Then no other size does as well.
- **Why not 1 MW either side?** A flat stretch narrower than 2 MW would pass that check.
- **For a max** (worst case, regret), take the branches at the max: the largest slope just above, the smallest just below.

In [ ]:
x_wc, z_wc, _ = solve(S, mode="worst")
x_rg, z_rg, _ = solve(S, mode="regret", ws=WSc)
for name, x, ref in (("worst case", x_wc, REF["rob"]), ("least regret", x_rg, REF["reg"])):
    _, ex, per = solve(S, fix=x)
    print(f"{name:12s}: {x:7.2f} MW, expected ${ex / 1e6:.2f}M, branch costs", {k_: round(float(v) / 1e6, 2) for k_, v in per.items()})
    assert abs(x - ref["x"]) < 1e-4 and abs(ex - ref["exp"]) <= 1
assert abs(z_rg - REF["reg"]["z"]) <= 1
def slope(W, x, side, a=A):
    # exact one-sided slope of cost(W, ., a) at x, $ per MW: side +1 just above x, -1 just below
    return c - cr * k * h[surplus(W, a) > x + side * 1e-6].sum()

def max_slope(pairs, x, side):
    # one-sided slope of the max over (W, offset) of cost(W, x) - offset, from the branches at the max (within $1)
    vals = [cost(W, x) - o for W, o in pairs]
    sl = [slope(W, x, side) for (W, o), v in zip(pairs, vals) if v >= max(vals) - 1]
    return max(sl) if side > 0 else min(sl)

for name, x, sl in (("stochastic", x_sp, lambda side: sum(s["p"] * slope(s["W"], x_sp, side) for s in S)),
                    ("worst case", x_wc, lambda side: max_slope([(s["W"], 0) for s in S], x_wc, side)),
                    ("least regret", x_rg, lambda side: max_slope([(s["W"], WSc[s["id"]]) for s in S], x_rg, side))):
    print(f"{name:12s}: slope just below {sl(-1):9,.0f}, just above {sl(+1):9,.0f} $ per MW")
    assert sl(-1) < 0 < sl(+1), name
print("Each plan is the only best size. The lab matches the browser.")

## 6. A bigger tree: two uncertainties, nine branches

A real study rarely has one uncertainty. Suppose what the lines take, $A$, is uncertain too (local demand may grow or shrink): 900, 1,000 or 1,100 MW, with chances 0.3, 0.4 and 0.3, independent of the build-out. Each branch is a pair, with chance $p_s \cdot q_a$, and the extensive form has $1 + 10 \times 9 = 91$ columns. Nothing below is asserted against the page: it is your model now, so only the order WS ≤ RP ≤ EEV is checked, and that the closed form, with each branch's own line limit, gives the LP's costs.

In [ ]:
A_levels = [(900, 0.3), (1000, 0.4), (1100, 0.3)]
S9 = [dict(id=f"{s['id']}-{a}", short=f"{s['short']}, A {a}", p=s["p"] * q, W=s["W"], A=a) for s in S for a, q in A_levels]
print(len(S9), "branches, chances add to", round(sum(s["p"] for s in S9), 6), "·", build(S9)["c"].size, "columns")
WS9 = sum(s["p"] * solve([dict(s, p=1)])[1] for s in S9)
mean_branch = dict(id="ev", short="average", p=1, W=sum(s["p"] * s["W"] for s in S9), A=sum(s["p"] * s["A"] for s in S9))
x_ev9 = solve([mean_branch])[0]
EEV9 = solve(S9, fix=x_ev9)[1]
x_sp9, RP9, _ = solve(S9)
print(f"stochastic {x_sp9:.1f} MW, plan for the average {x_ev9:.1f} MW")
print(f"WS ${WS9 / 1e6:.2f}M <= RP ${RP9 / 1e6:.2f}M <= EEV ${EEV9 / 1e6:.2f}M;  VSS ${(EEV9 - RP9) / 1e6:.2f}M, EVPI ${(RP9 - WS9) / 1e6:.2f}M")
assert WS9 <= RP9 + 1e-6 <= EEV9 + 2e-6
assert max(surplus(s["W"], s["A"]).max() for s in S9) > H / k, "with a 900 MW line limit the demand cap H/k binds"
# the closed form agrees with the LP: each branch priced with its own line limit
print(f"closed form: ${expected(S9, x_sp9) / 1e6:.2f}M at the stochastic plan, ${expected(S9, x_ev9) / 1e6:.2f}M at the plan for the average")
assert abs(expected(S9, x_sp9) - RP9) <= 1 and abs(expected(S9, x_ev9) - EEV9) <= 1

## 7. Practice (about 90 minutes)

Write your own cells below each task. Predict first, then solve, and report each result with the chances and the boundary it holds within.

1. **Uncertainties that move together** (about 20 minutes). In section 6 the line limit is independent of the build-out. Make it depend on it: with all the planned wind built, the lines take 1,100 MW with chance 0.6 (new lines came with the wind). Re-solve: which plans move, and do VSS and EVPI grow or shrink against the three-branch tree?
2. **An hourly year** (about 20 minutes). Replace the five blocks by the 8,760 hours of `wind` (each hour a block of one hour). How far does each plan move? Which plan is most sensitive to the time representation, and why?
3. **Risk** (about 20 minutes). The conditional value-at-risk (CVaR) is the average cost in the worst share of outcomes, here the worst 25%: the none-built branch. Minimize $(1 - \lambda)\,\text{expected cost} + \lambda\,\text{CVaR}$. The formulation and a starter are in the two cells after this list: finish `build_cvar`, then trace the size from $\lambda = 0$ (stochastic) to $\lambda = 1$.
4. **A second chance to build** (about 15 minutes). Let the planner add capacity $y_s$ after the build-out is known, at \$100,000 per MW a year. How much of the first-stage size moves to the second stage, and what does that do to EVPI?
5. **Sampled branches** (about 10 minutes). Draw 50 build-outs uniformly between 3,000 and 7,000 MW with a fixed seed, each with chance 1/50, and solve the stochastic program. Repeat with five seeds: how much does the plan scatter (M7's Monte Carlo error)?
6. **Write it up** (about 5 minutes). In five sentences, recommend a size: give its expected and worst cost, the chances and the yardstick you chose, and what the toy leaves out.

### Task 3, guided: CVaR as a linear program

**The idea (Rockafellar and Uryasev, 2000).** Pick a tail level $\alpha$; here $\alpha = 0.75$, so the tail is the worst 25% of outcomes. For a size $x$ with branch costs $\text{cost}_s(x)$,

$$\text{CVaR}_\alpha(x) = \min_{\eta}\; \eta + \frac{1}{1 - \alpha} \sum_s p_s \max\big(0,\; \text{cost}_s(x) - \eta\big).$$

- $\eta$, the **cutoff** (a free variable), chosen to make the expression smallest.
- **The cutoff need not be unique.** With a few discrete branches, a whole interval of cutoffs can give the same CVaR. At 845 MW, any $\eta$ from \$663.09M to \$731.49M does (the half-built and none-built costs). Read CVaR from the objective, and do not read the solver's $\eta$ as the value-at-risk.
- $u_s \ge 0$, the **excess cost** of branch $s$ above the cutoff: replace the max by $u_s \ge \text{cost}_s(x) - \eta$.

**The LP** adds $\eta$ and one $u_s$ per branch to the extensive form of section 3:

$$\min\; (1 - \lambda) \sum_s p_s\, \text{cost}_s + \lambda \Big(\eta + \frac{1}{1 - \alpha} \sum_s p_s u_s\Big)$$

$$\text{s.t.}\quad c\,x + \sum_b h_b c_r r_{s,b} - \eta - u_s \le 0 \;\text{ for every branch } s, \qquad u_s \ge 0,\; \eta \text{ free},$$

plus the hydrogen rows and the size rows of section 3. Here the none-built branch has chance 0.25 and costs most at every size, so at $\lambda = 1$ CVaR is that branch's cost: expect the worst-case plan. At $\lambda = 0$ it is the stochastic program, and the cutoff is not used at all.

Source: Rockafellar, R. T., and Uryasev, S. (2000). Optimization of conditional value-at-risk. *Journal of Risk* 2(3), 21–41, [doi:10.21314/JOR.2000.038](https://doi.org/10.21314/JOR.2000.038).

In [ ]:
# Starter for task 3: the columns and the objective are done; the excess-cost rows are yours (the TODO below).
alpha = 0.75

def build_cvar(S, lam, alpha=alpha):
    m = build(S)                                   # section 3's extensive form: column 0 is x, then e and r per branch and block
    n0, nS = m["c"].size, len(S)
    eta, u = n0, [n0 + 1 + i for i in range(nS)]   # new columns: the cutoff eta, then one excess u_s per branch
    cvec = np.concatenate([(1 - lam) * m["c"], np.zeros(1 + nS)])   # (1 - lam) x expected cost: m["c"] is its cost vector
    cvec[eta] = lam
    for i, s in enumerate(S):
        cvec[u[i]] = lam * s["p"] / (1 - alpha)
    bounds = m["bounds"] + [(-np.inf, np.inf)] + [(0, np.inf)] * nS
    A_eq = scipy.sparse.hstack([m["A_eq"], coo_matrix((m["A_eq"].shape[0], 1 + nS))]).tocsr()
    A_ub_old = scipy.sparse.hstack([m["A_ub"], coo_matrix((m["A_ub"].shape[0], 1 + nS))]).tocsr()
    # TODO: one row per branch s:  c x + sum_b h_b cr r_{s,b} - eta - u_s <= 0.
    #   Build them as a coo_matrix of shape (nS, n0 + 1 + nS), with m["col"](si, b, 1) for the column of r_{s,b};
    #   stack them under A_ub_old with scipy.sparse.vstack, and extend b_ub with nS zeros.
    A_ub, b_ub = A_ub_old, m["b_ub"]               # replace these two with your stacked rows
    return dict(c=cvec, A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=m["b_eq"], bounds=bounds)

def solve_cvar(S, lam):
    m = build_cvar(S, lam)
    res = linprog(m["c"], A_ub=m["A_ub"], b_ub=m["b_ub"], A_eq=m["A_eq"], b_eq=m["b_eq"], bounds=m["bounds"], method="highs")
    assert res.status == 0, res.message
    return res.x[0], res.fun

# Once your rows are in, trace the size:
# for lam in (0, 0.25, 0.5, 0.75, 1):
#     print(lam, round(solve_cvar(S, lam)[0], 1), "MW")

# The cutoff is not unique: at the stochastic plan, every cutoff between the half-built and none-built costs gives one CVaR.
cvar_at = lambda x, eta: eta + sum(s["p"] * max(0.0, cost(s["W"], x) - eta) for s in S) / (1 - alpha)
eta_lo, eta_hi = cost(S[1]["W"], x_sp), cost(S[0]["W"], x_sp)
assert all(abs(cvar_at(x_sp, e) - eta_hi) < 1e-3 for e in np.linspace(eta_lo, eta_hi, 7))
assert cvar_at(x_sp, eta_lo - 1e6) > eta_hi + 1 and cvar_at(x_sp, eta_hi + 1e6) > eta_hi + 1
print(f"at {x_sp:.0f} MW every cutoff from ${eta_lo / 1e6:.2f}M to ${eta_hi / 1e6:.2f}M gives the same CVaR")